# Theta phase precession from button presses

This notebook estimates theta phase precession from the UPPT002 button-press times and the MEG epochs. It filters each MEG sensor to 2–5 Hz, extracts phase with MNE's Hilbert transform, then uses circ_regress_db to fit a circular-linear slope and signed correlation for each trial and sensor.

Trial results are averaged within each participant before the participant means are combined. Subject 11 is excluded. Only trials with all five valid press times are included. The group permutation tests use participant means and apply false-discovery-rate correction across sensors; they are not the per-trial phase-shuffle tests from circ_regress_db. Each trial contributes only five phase samples, so individual fits may be unstable. Use the slope for phase-drift direction: the circular-correlation coefficient's sign does not always match the slope.

import packages for loading and processing MEG epochs, circular regression, tables and figures

In [ ]:
# Load the tools used for MNE epochs, array processing, result tables and plots.
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from circ_regress_db import circ_regress_db

define the data locations and analysis parameters

data_root can point to a local MEG export; otherwise the notebook uses the repository's network-share path. The button-press file is expected in the repository folder.

theta_freq_band sets the filter limits. Subject 11 is excluded before analysis. channel_batch_size limits memory use while channels are filtered and Hilbert-transformed. n_group_permutations sets the number of permutations for the participant-level tests.

In [ ]:
# Use a local data export when configured; otherwise use the shared repository location.
data_root = os.environ.get(
    "MEG_DATA_ROOT",
    r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
)
# Keep the timing file alongside this notebook.
button_press_times_file = os.path.join(
    os.getcwd(), "UPPT002_button_press_times.npz"
)

# Match the 2–5 Hz theta band used in the related MEG notebooks.
theta_freq_band = [2.0, 5.0]             # theta band edges in Hz
excluded_subjects = ["11"]               # participants excluded from analysis
channel_batch_size = 8                   # channels loaded for each MNE processing batch
n_group_permutations = 5000              # group-level permutation tests

load and check the UPPT002 button-press times

The NPZ file stores uppt002_times in participant × trial × press-slot order. subject_ID records the participant row order, n_trials_by_subject gives each participant's recorded trial count, and press_slot labels presses 1 through 5. Times are in seconds relative to cue onset.

The ID helper normalizes participant labels to the repository's two-digit format. The checks confirm the time reference, array dimensions and press-slot order before the participant list is made. Subject 11 is omitted from sub_ID here, so it is not included in the per-participant or group calculations.

In [ ]:
# Match IDs from the NPZ file to the two-digit labels used by the MEG notebooks.
def normalise_subject_id(id):
    return str(int(str(id).strip().lower().lstrip("s"))).zfill(2)


# Read the timing arrays and their metadata from the button-press export.
with np.load(button_press_times_file, allow_pickle=False) as press_file:
    UPPT002_times = np.asarray(press_file["uppt002_times"], dtype=float)
    subject_ID = np.asarray(
        [normalise_subject_id(id) for id in press_file["subject_ID"]]
    )
    n_trials_by_subject = np.asarray(
        press_file["n_trials_by_subject"], dtype=int
    )
    press_slot = np.asarray(press_file["press_slot"], dtype=int)
    cue_onset_after_epoch_start_s = float(
        np.asarray(press_file["cue_onset_after_epoch_start_s"]).item()
    )
    time_reference = str(press_file["time_reference"].item())

# Check that the saved times use the expected reference and array layout.
if "relative to cue onset" not in time_reference.lower():
    raise ValueError("Expected button-press times in seconds relative to cue onset")
if UPPT002_times.ndim != 3 or UPPT002_times.shape[0] != len(subject_ID):
    raise ValueError("Unexpected participant × trial × press-slot array shape")
if not np.array_equal(press_slot, np.arange(1, len(press_slot) + 1)):
    raise ValueError("Expected press slots numbered in sequence from 1")

# Keep repository-style subject labels and a lookup to each participant's NPZ row.
n_press_slots = len(press_slot)
sub_ID = [id for id in subject_ID if id not in excluded_subjects]
subject_index = {id: index for index, id in enumerate(subject_ID)}
if not sub_ID:
    raise ValueError("No participants remain after applying excluded_subjects.")

print("Array shape:", UPPT002_times.shape)
print("Participants:", sub_ID)
print("Excluded participants:", excluded_subjects)

load each participant's MEG epochs

filename_loader follows the repository layout: epochs are under sXX and the filename uses the participant number without its leading zero. It also checks for a local export with the FIF file at the data-root level.

The epochs are opened without loading all channels into memory. MEG_sens contains channels whose names begin with M. mean_finite averages finite values along the requested axis and returns NaN when no values are available.

In [ ]:
# Locate the participant's saved epochs using the repository's subject-folder naming.
def filename_loader(id):
    subject_number = str(int(id))
    subject_folder = "s" + id
    file_name = "Subject" + subject_number + "_spm_mne-epo.fif"
    epochs_file = os.path.join(data_root, subject_folder, file_name)

    # Also allow a local export with the FIF file directly in data_root.
    if not os.path.isfile(epochs_file):
        epochs_file = os.path.join(data_root, file_name)
    if not os.path.isfile(epochs_file):
        raise FileNotFoundError("Could not find " + file_name + " under " + data_root)

    # Keep the epochs file-backed; selected channel batches are read when needed.
    epochs = mne.read_epochs(epochs_file, preload=False, verbose="ERROR")
    MEG_sens = [channel for channel in epochs.ch_names if channel.startswith("M")]
    if not MEG_sens:
        raise ValueError("No MEG sensors found in " + epochs_file)

    return epochs, MEG_sens


# Average only finite values and preserve NaN when an axis has no valid values.
def mean_finite(values, axis=0):
    values = np.asarray(values)
    counts = np.isfinite(values).sum(axis=axis)
    return np.divide(
        np.nansum(values, axis=axis),
        counts,
        out=np.full(counts.shape, np.nan, dtype=np.result_type(values.dtype, float)),
        where=counts > 0,
    )

calculate theta phase precession for one participant

The function checks that the NPZ timing rows match the saved MEG epochs, maps cue-relative press times onto the epoch time axis, and marks trials with all five valid press times.

For each small sensor batch, get_data reads only the requested channels from the non-preloaded epochs. The batch is wrapped in an EpochsArray with matching sensor information; MNE then filters it to theta and applies the Hilbert transform. This keeps the original epochs on disk and loads only the current channel batch into memory.

The phase at each press is passed to circ_regress_db only when all five phase samples are finite. The helper fits the slope and phase offset for the press-slot sequence and returns the circular-correlation coefficient. Its sign does not always match the slope, so use the slope to interpret phase-drift direction. The function averages slope and correlation across valid trials, averages phase offsets circularly, and returns sensor geometry and trial counts for review.

In [ ]:
def participant_phase_precession(id):
    # Load this participant's epochs and find their row in the timing file.
    epochs, MEG_sens = filename_loader(id)
    data_index = subject_index[id]
    n_trials = int(n_trials_by_subject[data_index])

    # Require one timing row for every saved MEG epoch.
    if len(epochs) != n_trials:
        raise ValueError(
            "Participant " + id + " has " + str(len(epochs))
            + " MEG epochs but " + str(n_trials) + " timing rows"
        )

    # Convert cue-relative press times to MNE epoch sample indices.
    press_times = UPPT002_times[data_index, :n_trials]
    cue_onset_epoch_s = epochs.times[0] + cue_onset_after_epoch_start_s
    event_times = press_times + cue_onset_epoch_s
    sample_indices = np.zeros(press_times.shape, dtype=int)
    valid_press = np.isfinite(event_times)
    sample_indices[valid_press] = np.rint(
        (event_times[valid_press] - epochs.times[0]) * epochs.info["sfreq"]
    ).astype(int)
    valid_press &= (
        (sample_indices >= 0) & (sample_indices < len(epochs.times))
    )
    # Use trials only when all five press times fall inside their MEG epoch.
    complete_trial = valid_press.sum(axis=1) == n_press_slots

    # Store one fitted slope, phase offset and correlation per trial and sensor.
    trial_slope = np.full((n_trials, len(MEG_sens)), np.nan)
    trial_phase = np.full_like(trial_slope, np.nan)
    trial_corr = np.full_like(trial_slope, np.nan)

    # Read and transform a small channel batch at a time to limit memory use.
    for start in range(0, len(MEG_sens), channel_batch_size):
        channel_batch = MEG_sens[start:start + channel_batch_size]
        channel_indices = mne.pick_channels(
            epochs.ch_names, include=channel_batch, ordered=True
        )
        channel_data = epochs.get_data(picks=channel_indices)
        batch_info = mne.pick_info(epochs.info, channel_indices, copy=True)

        # Wrap the loaded data with matching sensor metadata for MNE processing.
        epochs_batch = mne.EpochsArray(
            channel_data,
            batch_info,
            events=epochs.events,
            tmin=epochs.tmin,
            baseline=None,
            verbose="ERROR",
        )
        # Filter to theta and form the complex analytic signal to extract phase.
        epochs_batch.filter(
            l_freq=theta_freq_band[0],
            h_freq=theta_freq_band[1],
            verbose="ERROR",
        ).apply_hilbert(envelope=False, verbose="ERROR")
        analytic = epochs_batch.get_data()

        # Fit the circular-linear model separately for each complete trial/sensor.
        for trial in np.flatnonzero(complete_trial):
            samples = sample_indices[trial]
            for channel_index in range(len(channel_batch)):
                # np.angle returns radians in [-pi, pi], equivalent modulo 2*pi.
                phase = np.angle(analytic[trial, channel_index, samples])
                # The helper drops NaNs internally, so require all five samples here.
                if not np.isfinite(phase).all():
                    continue
                result = circ_regress_db(press_slot, phase, k=0, ppdir=0)
                # The slope gives direction; correlation sign may differ from slope.
                trial_slope[trial, start + channel_index] = result["slope_opt"]
                trial_phase[trial, start + channel_index] = result["phase_opt"]
                trial_corr[trial, start + channel_index] = result["corr"]

        # Release this batch before loading the next set of channels.
        del analytic
        del epochs_batch
        del channel_data

    # Average trial results within this participant; wrap phase offsets circularly.
    participant_slope = mean_finite(trial_slope, axis=0)
    participant_phase = np.angle(
        mean_finite(np.exp(1j * trial_phase), axis=0)
    )
    participant_corr = mean_finite(trial_corr, axis=0)

    # Return channel geometry plus counts so group processing can verify alignment.
    picks = mne.pick_channels(epochs.ch_names, include=MEG_sens, ordered=True)
    sensor_info = mne.pick_info(epochs.info, picks, copy=True)
    summary = {
        "participant": id,
        "n_trials": n_trials,
        "n_complete_press_trials": int(complete_trial.sum()),
        "n_valid_press_times": int(valid_press.sum()),
        "n_trials_with_corr": int(np.isfinite(trial_corr).any(axis=1).sum()),
    }

    return (
        participant_slope, participant_corr, participant_phase,
        MEG_sens, sensor_info, summary
    )

process the included participants

sub_ID already excludes subject 11. Each participant is analyzed separately, and the sensor names and order are checked before values are combined. This prevents channel values from being averaged under mismatched sensor labels.

The group values are means of the participant means, so participants with more valid trials do not receive extra weight. Phase offsets are averaged as circular values.

In [ ]:
# Collect each participant's sensor summaries after excluding subject 11.
participant_slope = []
participant_corr = []
participant_phase = []
participant_rows = []
group_sensors = None
group_info = None

for id in sub_ID:
    slope, corr, phase, MEG_sens, sensor_info, summary = (
        participant_phase_precession(id)
    )
    # Keep sensor order fixed so every participant contributes to the same columns.
    if group_sensors is None:
        group_sensors, group_info = MEG_sens, sensor_info
    elif MEG_sens != group_sensors:
        raise ValueError("MEG sensor order differs for participant " + id)

    # Save participant-level values and report the number of timing-complete trials.
    participant_slope.append(slope)
    participant_corr.append(corr)
    participant_phase.append(phase)
    participant_rows.append(summary)
    print(
        "Participant", id,
        "- complete trials:", summary["n_complete_press_trials"],
    )

# Rows now represent participants and columns represent MEG sensors.
participant_slope = np.asarray(participant_slope)
participant_corr = np.asarray(participant_corr)
participant_phase = np.asarray(participant_phase)
participant_summary_table = pd.DataFrame(participant_rows)

# Calculate equal-weight group means; phase offsets require a circular mean.
group_slope = mean_finite(participant_slope, axis=0)
group_corr = mean_finite(participant_corr, axis=0)
group_phase = np.angle(
    mean_finite(np.exp(1j * participant_phase), axis=0)
)

test participant means and review the results

For each sensor, group_permutation_test runs a two-sided one-sample permutation test across participant means against zero. The FDR correction is applied across sensors separately for correlation and slope.

The tables show participant-level sensor values, participant trial counts, group means and corrected p-values. These p-values describe group-level effects across participants; they are not the per-trial phase-shuffle p-values returned by circ_regress_db when k is greater than zero.

In [ ]:
# Test participant-level sensor means against zero; apply FDR across sensors.
def group_permutation_test(values):
    t_value = np.full(values.shape[1], np.nan)
    p_value = np.full(values.shape[1], np.nan)
    rng = np.random.RandomState(0)

    # Run a one-sample permutation test for each sensor with enough participants.
    for sensor_index in range(values.shape[1]):
        sensor_values = values[:, sensor_index]
        sensor_values = sensor_values[np.isfinite(sensor_values)]
        if len(sensor_values) < 3:
            continue

        t_value[sensor_index], p_value[sensor_index], _ = (
            mne.stats.permutation_t_test(
                sensor_values[:, None],
                n_permutations=n_group_permutations,
                tail=0,
                seed=rng,
                verbose="ERROR",
            )
        )

    # Correct the resulting sensor-wise p-values for multiple comparisons.
    p_fdr = np.full_like(p_value, np.nan)
    valid = np.isfinite(p_value)
    if valid.any():
        _, p_fdr[valid] = mne.stats.fdr_correction(p_value[valid], alpha=0.05)

    return t_value, p_value, p_fdr


# These tests use participant means; per-trial helper p-values are not calculated.
corr_t, corr_p, corr_p_fdr = group_permutation_test(participant_corr)
slope_t, slope_p, slope_p_fdr = group_permutation_test(participant_slope)

# Organize participant-level values and group statistics for inspection.
participant_corr_table = pd.DataFrame(
    participant_corr, index=sub_ID, columns=group_sensors
)
participant_slope_table = pd.DataFrame(
    participant_slope, index=sub_ID, columns=group_sensors
)
participant_phase_table = pd.DataFrame(
    participant_phase, index=sub_ID, columns=group_sensors
)

group_results = pd.DataFrame({
    "sensor": group_sensors,
    "mean_corr": group_corr,
    "corr_t": corr_t,
    "corr_p": corr_p,
    "corr_p_fdr": corr_p_fdr,
    "mean_slope_rad_per_press": group_slope,
    "slope_t": slope_t,
    "slope_p": slope_p,
    "slope_p_fdr": slope_p_fdr,
    "mean_phase_offset_rad": group_phase,
})

# Display the participant summaries and corrected group results.
display(participant_summary_table.round(4))
display(participant_corr_table.round(3))
display(participant_slope_table.round(3))
display(group_results.round(4))

plot the group mean for each MEG sensor type

The correlation and slope maps show the equal-weight mean across participant means. Magnetometers and gradiometers are plotted separately using the channel geometry loaded from the FIF epochs.

The displayed color limits span the full coefficient or slope range. Use group_results to review the corresponding uncorrected and FDR-corrected sensor-level p-values.

In [ ]:
# Use the MNE sensor metadata to split maps by magnetometer/gradiometer type.
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

# Plot the group mean correlation and slope for each sensor type.
for values, label, limits in [
    (group_corr, "Mean signed circular correlation", (-1.0, 1.0)),
    (group_slope, "Mean slope (radians per press)", (-2 * np.pi, 2 * np.pi)),
]:
    for sensor_type in sorted(set(sensor_types)):
        picks = np.array([
            index for index, kind in enumerate(sensor_types)
            if kind == sensor_type
        ])
        sensor_info = mne.pick_info(group_info, picks, copy=True)
        fig, ax = plt.subplots(figsize=(7, 6))
        image, _ = mne.viz.plot_topomap(
            values[picks],
            sensor_info,
            ch_type=sensor_type,
            axes=ax,
            cmap="RdBu_r",
            contours=0,
            show=False,
        )
        # Apply one fixed colour range per measure for comparable sensor maps.
        image.set_clim(*limits)
        ax.set_title(label + " (" + sensor_type + ")")
        fig.colorbar(image, ax=ax)
        plt.tight_layout()
        plt.show()